In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('customer_propensity_processed_data.csv')
X = dataset.drop("purchased", axis=1)
y = dataset["purchased"]

In [3]:
dataset.head()

,gender_Female,gender_Male,marital_status_Divorced,marital_status_Married,marital_status_Single,marital_status_Widowed,education_Bachelor,education_Diploma,education_High School,education_Master,...,days_since_last_purchase,returned_orders,purchased,purchase_frequency,avg_monthly_spend,cart_abandonment_rate,purchase_conversion_rate,browsing_to_purchase_ratio,rfm_score,age_group
0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,128.0,1.0,0,0.86,14567.65,0.038,1.333,2.52,14,30
1,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,164.0,0.0,0,0.18,109.88,0.600,0.143,29.00,4,20
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,74.0,5.0,1,1.20,5717.82,0.032,2.458,1.65,15,40
3,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,162.0,0.0,0,0.25,799.07,0.250,0.111,21.33,6,40
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,145.0,0.0,0,0.38,1189.47,0.091,0.760,5.70,9,40


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [5]:
from sklearn.preprocessing import StandardScaler

numeric_features = [
    'age_group',
    'website_visits_last_30d',
    'app_sessions_last_30d',
    'avg_session_duration',
    'product_views',
    'categories_viewed',
    'search_count',
    'wishlist_items',
    'total_orders',
    'average_order_value',
    'total_amount_spent',
    'months_active',
    'days_since_last_purchase',
    'returned_orders',
    'discount_percent',
    'campaign_duration_days',
    'campaign_send_hour',
    'campaign_cost',
    'previous_campaigns_received',
    'previous_campaign_response_rate',
    'annual_income',
    'purchase_frequency',
    'purchase_conversion_rate',
    'browsing_to_purchase_ratio',
    'avg_monthly_spend',
    'cart_abandonments',
    'rfm_score'
]

scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

In [6]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)

X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

In [7]:
from imblearn.combine import SMOTEENN
smote = SMOTEENN(sampling_strategy = 0.6777, random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

C:\Users\Nafees\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Nafees\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\Nafees\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nafees\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^

In [8]:
from xgboost import XGBClassifier
classifier = XGBClassifier(n_estimators = 100, learning_rate = 0.1, max_depth = 11, random_state = 0)
classifier.fit(X_train_smote, y_train_smote)

AttributeError: 'super' object has no attribute '__sklearn_tags__'

AttributeError: 'super' object has no attribute '__sklearn_tags__'

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=11, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=0, ...)

In [9]:
y_pred = classifier.predict(X_test)

In [10]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[7173 1858]
 [ 417 3072]]


0.8182907348242812

In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      9031
           1       0.62      0.88      0.73      3489

    accuracy                           0.82     12520
   macro avg       0.78      0.84      0.80     12520
weighted avg       0.86      0.82      0.83     12520



In [12]:
y_test.value_counts()

purchased
0    9031
1    3489
Name: count, dtype: int64